# 21B · Gray-Box AIA72 Real-Outage Structure Audit

Phase E2 reconstructs empirical AIA outage episodes from the retained 2025–2026 recovery inventories, including rows whose original `block_id` is blank. It asks whether observed acquisition failures are isolated or temporally/regionally clustered.

Episode reconstruction is descriptive only. It does not alter the frozen Gray-Box policy, and it does not convert engineering acquisition failures into claims about historical operational outages.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd

ROOT=Path('/home/abmoses2000')
SRC=ROOT/'graybox_aia72_real_outage_replay_v1_20261006'
INC=SRC/'real_acquisition_incidents.csv'
MAP=SRC/'incident_to_graybox_mapping.csv'
REPLAY=SRC/'real_outage_replay_summary.csv'

OUT=ROOT/'graybox_aia72_real_outage_structure_v1_20261006'
if OUT.exists():
    raise SystemExit(f'Refusing to overwrite existing output: {OUT}')
OUT.mkdir(parents=True)

for p in [INC,MAP,REPLAY]:
    if not p.is_file():
        raise ValueError(f'Missing Phase E1 artifact: {p}')

inc=pd.read_csv(INC,low_memory=False)
mapped=pd.read_csv(MAP,low_memory=False)

required={'sample_id','T_REC_dt','HARPNUM','recovery_class','block_id','year'}
if not required.issubset(inc.columns):
    raise ValueError(f'Incident schema changed; missing {required-set(inc.columns)}')

inc['T_REC_dt']=pd.to_datetime(inc['T_REC_dt'],utc=True,errors='coerce')
if inc.T_REC_dt.isna().any():
    raise ValueError('Unparseable incident timestamps')

# ------------------------------------------------------------
# Episode reconstruction
# ------------------------------------------------------------
# Native forecast/sample cadence is 96 min. To avoid splitting modestly irregular
# sequences, declare a new empirical episode only when the same HARP + recovery
# class gap exceeds 192 min (two cadence steps). This is descriptive, not tuned.
CADENCE_MIN=96
GAP_BREAK_MIN=192

inc=inc.sort_values(['year','HARPNUM','recovery_class','T_REC_dt','sample_id']).reset_index(drop=True)

episode_ids=[]
episode_n=0
prev_key=None
prev_time=None
for r in inc.itertuples():
    key=(int(r.year),str(r.HARPNUM),str(r.recovery_class))
    new=(key!=prev_key)
    if not new and prev_time is not None:
        gap=(r.T_REC_dt-prev_time).total_seconds()/60
        if gap>GAP_BREAK_MIN:
            new=True
    if new:
        episode_n+=1
    episode_ids.append(f'E{episode_n:05d}')
    prev_key=key
    prev_time=r.T_REC_dt

inc['empirical_episode_id']=episode_ids

# Preserve explicit block IDs when present, but never assume blank IDs mean isolated events.
inc['has_explicit_block_id']=inc['block_id'].fillna('').astype(str).str.len().gt(0)

ep=inc.groupby(['empirical_episode_id','year','HARPNUM','recovery_class'],as_index=False).agg(
    samples=('sample_id','size'),
    first_time=('T_REC_dt','min'),
    last_time=('T_REC_dt','max'),
    explicit_block_ids=('block_id',lambda x:';'.join(sorted({str(v) for v in x if pd.notna(v) and str(v)}))),
    explicit_block_rows=('has_explicit_block_id','sum'),
)
ep['span_hours']=(ep.last_time-ep.first_time).dt.total_seconds()/3600
ep['cadence_steps_equivalent']=np.where(ep.samples>1,(ep.span_hours*60/CADENCE_MIN)+1,1.0)
ep['is_clustered']=ep.samples>1
ep['has_explicit_block_id']=ep.explicit_block_ids.str.len().gt(0)

# Link mapped Gray-Box cases.
if len(mapped):
    mm=mapped[['sample_id','row_index','persistent_after_recovery']].copy()
    inc=inc.merge(mm,on='sample_id',how='left',validate='one_to_one')
    inc['maps_to_graybox']=inc.row_index.notna()
else:
    inc['maps_to_graybox']=False
    inc['persistent_after_recovery']=False

ep_map=inc.groupby('empirical_episode_id').agg(
    mapped_graybox_cases=('maps_to_graybox','sum'),
    persistent_mapped_cases=('persistent_after_recovery','sum')
).reset_index()
ep=ep.merge(ep_map,on='empirical_episode_id',how='left',validate='one_to_one')

inc.to_csv(OUT/'incident_episode_assignments.csv',index=False)
ep.to_csv(OUT/'empirical_outage_episodes.csv',index=False)

# ------------------------------------------------------------
# Summaries
# ------------------------------------------------------------
summary=[]
for year,g in ep.groupby('year',sort=True):
    summary.append({
        'year':int(year),
        'episodes':len(g),
        'clustered_episodes':int(g.is_clustered.sum()),
        'clustered_episode_fraction':float(g.is_clustered.mean()),
        'samples_in_clustered_episodes':int(g.loc[g.is_clustered,'samples'].sum()),
        'total_incident_samples':int(g.samples.sum()),
        'median_episode_samples':float(g.samples.median()),
        'p90_episode_samples':float(g.samples.quantile(.90)),
        'max_episode_samples':int(g.samples.max()),
        'median_episode_span_hours':float(g.span_hours.median()),
        'p90_episode_span_hours':float(g.span_hours.quantile(.90)),
        'max_episode_span_hours':float(g.span_hours.max()),
        'episodes_with_explicit_block_id':int(g.has_explicit_block_id.sum()),
        'mapped_graybox_cases':int(g.mapped_graybox_cases.sum()),
    })
summary=pd.DataFrame(summary)
summary.to_csv(OUT/'episode_structure_by_year.csv',index=False)

class_summary=ep.groupby(['year','recovery_class']).agg(
    episodes=('empirical_episode_id','size'),
    clustered_episodes=('is_clustered','sum'),
    incident_samples=('samples','sum'),
    median_samples_per_episode=('samples','median'),
    max_samples_per_episode=('samples','max'),
    median_span_hours=('span_hours','median'),
    max_span_hours=('span_hours','max'),
    mapped_graybox_cases=('mapped_graybox_cases','sum')
).reset_index()
class_summary.to_csv(OUT/'episode_structure_by_class.csv',index=False)

top=ep.sort_values(['samples','span_hours'],ascending=False).head(30)
top.to_csv(OUT/'largest_outage_episodes.csv',index=False)

# Compare explicit block IDs against reconstructed episodes only where block IDs exist.
blk=inc[inc.has_explicit_block_id].copy()
block_reconciliation=[]
if len(blk):
    for bid,g in blk.groupby('block_id'):
        block_reconciliation.append({
            'block_id':bid,
            'rows':len(g),
            'empirical_episodes':int(g.empirical_episode_id.nunique()),
            'harps':int(g.HARPNUM.nunique()),
            'recovery_classes':int(g.recovery_class.nunique()),
            'first_time':g.T_REC_dt.min(),
            'last_time':g.T_REC_dt.max(),
        })
pd.DataFrame(block_reconciliation).to_csv(OUT/'explicit_block_reconciliation.csv',index=False)

# ------------------------------------------------------------
# UQ / uncertainty context for outage episodes
# ------------------------------------------------------------
# E2 does not fit UQ. It carries forward the already-established uncertainty layer:
# probability calibration, Mondrian conformal coverage, seed disagreement, entropy,
# and fallback/abstention state logic. Here we quantify how many mapped incident
# cases belong to outage episodes of increasing severity, so robustness can be
# interpreted alongside the frozen UQ framework.
severity=[]
if len(mapped):
    z=inc[inc.maps_to_graybox].merge(ep[['empirical_episode_id','samples','span_hours','is_clustered']],
                                    on='empirical_episode_id',how='left',suffixes=('','_ep'))
    for label,mask in [
        ('isolated',z.samples==1),
        ('short_cluster_2_4',z.samples.between(2,4)),
        ('medium_cluster_5_10',z.samples.between(5,10)),
        ('long_cluster_gt10',z.samples>10),
    ]:
        h=z[mask]
        severity.append({
            'episode_severity':label,
            'mapped_graybox_cases':len(h),
            'unique_episodes':int(h.empirical_episode_id.nunique()),
            'persistent_cases':int(pd.to_numeric(h.persistent_after_recovery,errors='coerce').fillna(False).astype(bool).sum()),
            'median_episode_span_hours':float(h.span_hours.median()) if len(h) else np.nan,
        })
pd.DataFrame(severity).to_csv(OUT/'graybox_exposure_by_episode_severity.csv',index=False)

protocol={
    'status':'POSTHOC_REAL_AIA_OUTAGE_STRUCTURE_AUDIT_COMPLETE',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'horizon_hours':72,
    'new_model_fitting':False,
    'new_threshold_tuning':False,
    'new_policy_selection':False,
    'new_policy_redesign':False,
    'native_cadence_minutes':CADENCE_MIN,
    'episode_rule':f'Within same year+HARPNUM+recovery_class, start a new episode when inter-failure gap exceeds {GAP_BREAK_MIN} minutes.',
    'episode_rule_status':'descriptive deterministic grouping; not optimized against forecast outcomes',
    'uq_context':'No new UQ fitted here. Interpretation remains tied to the frozen calibration, Mondrian conformal, seed-disagreement, entropy, and fallback/abstention framework.',
    'limitations':[
        'Engineering acquisition/recovery incidents are not equivalent to verified live-service outages.',
        'Blank original block_id values are not treated as independent events.',
        'Episode reconstruction uses temporal/HARP structure only and does not infer silent corruption.',
        'No policy thresholds are changed using these episodes.'
    ],
    'outputs':['incident_episode_assignments.csv','empirical_outage_episodes.csv','episode_structure_by_year.csv',
               'episode_structure_by_class.csv','largest_outage_episodes.csv','explicit_block_reconciliation.csv',
               'graybox_exposure_by_episode_severity.csv']
}
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')

print('===== REAL AIA OUTAGE STRUCTURE AUDIT COMPLETE =====')
print('\nBy year:')
print(summary.round(4).to_string(index=False))
print('\nBy class:')
print(class_summary.round(4).to_string(index=False))
print('\nLargest episodes:')
print(top[['empirical_episode_id','year','HARPNUM','recovery_class','samples','span_hours','mapped_graybox_cases']].to_string(index=False))
print('\nOutputs:',OUT)


===== REAL AIA OUTAGE STRUCTURE AUDIT COMPLETE =====

By year:
 year  episodes  clustered_episodes  clustered_episode_fraction  samples_in_clustered_episodes  total_incident_samples  median_episode_samples  p90_episode_samples  max_episode_samples  median_episode_span_hours  p90_episode_span_hours  max_episode_span_hours  episodes_with_explicit_block_id  mapped_graybox_cases
 2025       359                 332                      0.9248                           6172                    6199                    13.0                 44.2                   78                       19.2                   69.44                   124.8                               32                  2289
 2026        18                  18                      1.0000                            262                     262                    13.5                 30.0                   30                       20.3                   46.40                    46.4                               18               

## Interpretation guardrails

- `empirical_episode_id` is a deterministic descriptive grouping, not a recovered original production block ID.
- The 192-minute break rule is based on the 96-minute cadence and is fixed before inspecting forecast outcomes.
- UQ remains part of the broader Gray-Box architecture: this notebook does not replace the existing conformal / calibration / disagreement analyses.
